In [1]:

import pandas as pd
import torch
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

# Глянем на датасет

In [2]:


# загрузка датасета
raw = pd.read_csv('../data/yelp_reviews.csv')

texts = raw['text']
labels = raw['label']


print('\nРазмер датасета:')
print(len(texts))


print('\nПервый отзыв из датасета:')
print('  '.join(texts[0].split('\\n')))


print('\nИ его рейтинг:')
print(labels[0])


Размер датасета:
6500

Первый отзыв из датасета:
Worst sandwich on Earth.  I'd rather eat a dead whore.  Please...never come here.

И его рейтинг:
0


# Задание 1
Что нужно сделать
* Подготовьте датасет. 
* Чтобы сформировать батчи, используйте кастомную функцию collate_fn. 
* Подготовка датасета проводится аналогично подготовке датасета для токенизации текста. 
* При инициализации датасета нужно сохранить аргументы из конструктора, делать дополнительные преобразования не нужно.
* В методе `__getitem__` нужно возвращать объекты класса `torch.tensor`. 
* Текст можно обрезать обычной срезкой списка в питоне `[:self.max_len]`.
* В кастомной функции `collate_fn` для пэддинга используйте метод `pad_sequence`.

In [ ]:
# разделение выборки на трейн и тест
train_texts, val_texts, train_labels, val_labels = train_test_split(
                                                   texts, labels, test_size=0.2, random_state=42)


# создание токенизатора с помощью класса AutoTokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


# токенизируем тексты
train_texts_tokenized = tokenizer(train_texts.tolist(), truncation=True)['input_ids']
val_texts_tokenized = tokenizer(val_texts.tolist(), truncation=True)['input_ids']


# создаём класс кастомного датасета, наследуясь от класса Dataset из PyTorch

class YelpDataset(Dataset):
    # в конструкторе просто сохраняем тексты и классы
    def __init__(self, texts, labels, max_len=256):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len


    # возвращаем размер датасета (кол-во текстов)
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        # возвращаем текст и его класс
        # для текста ограничиваем длину
        # не делаем никаких доп. преобразований как padding и masking
        return {
            'text': torch.tensor(self.texts[idx][:self.max_len], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }


# кастомная функция collate_fn для формирования батчей
def collate_fn(batch):
    texts = [torch.tensor(item['text']) for item in batch]
    labels = torch.tensor([item['label'] for item in batch])
    lengths = torch.tensor([len(seq) for seq in texts])
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)


    return {
        'input_ids': padded_texts, 
        'lengths': lengths, 
        'labels': labels
    }


train_dataset = YelpDataset(texts=train_texts_tokenized, labels=train_labels.tolist())
val_dataset = YelpDataset(texts=val_texts_tokenized, labels=val_labels.tolist())


batch_size = 64


train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

Количество батчей в train_dataloader: 82
Количество батчей в val_dataloader: 21
Размерности батчей:
input_ids: torch.Size([64, 256])
lengths: torch.Size([64])
labels: torch.Size([64])


C:\Users\aseva\AppData\Local\Temp\ipykernel_7112\3247054542.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(item['text']) for item in batch]


In [ ]:

print(f'Количество батчей в train_dataloader: {len(train_dataloader)}')
print(f'Количество батчей в val_dataloader: {len(val_dataloader)}')


print('Размерности батчей:')
for batch in train_dataloader:
    print('input_ids:', batch['input_ids'].shape)
    print('lengths:', batch['lengths'].shape)
    print('labels:', batch['labels'].shape)
    break